# Ouroboros — v1 grid on Colab

Runs the same 36-task grid as the Hoffman2 array (`scripts/hpc/v1_grid.py`): arms Babs/Brel/C x real-label budget K in {0,128,512,2048} x seeds 0-2. Uses the synthetic shards already on Drive (`ouroboros/data/full/shards`) and rebuilds the real splits from the Hugging Face dataset (CC BY-NC; stays on this VM, never committed). Each finished run is written to Drive (`ouroboros/v1/runs/<arm>_K<k>_s<seed>`); rerunning the loop skips finished runs and resumes interrupted ones from `last.pt`.

Colab's system Python (3.13) cannot install the pinned stack (escnn needs lie_learn and numpy<2, which have no 3.13 builds), so the setup creates a Python 3.11 virtualenv with `uv` at `/content/venv` and installs exactly `requirements.txt` (incl. torch 2.10.0 CUDA). All project code runs in that interpreter via subprocesses (`!{PY} ...`). If the repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repository.

In [ ]:
import time, os, subprocess, json
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true
!python --version && nproc && free -g | head -2
VENV_PY = '/content/venv/bin/python'  # project interpreter (created below)
# Colab's kernel exports MPLBACKEND=module://matplotlib_inline..., which does not exist in
# the venv; every project command therefore runs with an explicit headless backend.
PY = f'MPLBACKEND=Agg {VENV_PY}'

In [ ]:
# ---- configuration ----
REPO = 'yaniguan/ouroboros-ocsr'
BRANCH = 'codex/v1-multiseed-cplus'  # set to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/ouroboros'  # data, runs, results live here
REPO_DIR = '/content/ouroboros-ocsr'
LOCAL_DATA = '/content/data/full'  # configs expect shards at /content/data/full/shards
# the kernel's inline matplotlib backend does not exist in the project venv (subprocesses)
os.environ['MPLBACKEND'] = 'Agg'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for sub in ('data', 'runs', 'results', 'real'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log --oneline -1

In [ ]:
%%bash -s "$REPO_DIR"
set -e
cd "$1"
# py3nj (escnn dependency) builds from source and needs a Fortran compiler
which gfortran || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
pip install -q uv
# Python 3.11 venv with the exact tested pins (Colab's system Python is 3.13)
[ -x /content/venv/bin/python ] || uv venv -q --python 3.11 --python-preference only-managed /content/venv
uv pip install -q --python /content/venv/bin/python -r requirements.txt
uv pip install -q --python /content/venv/bin/python --no-deps -e .
# startup hook: any python of this venv replaces an inherited inline matplotlib backend
SP=$(/content/venv/bin/python -c 'import site; print(site.getsitepackages()[0])')
cp scripts/colab_mplbackend.pth "$SP/"
export MPLBACKEND=Agg
/content/venv/bin/python -c "import sys, torch; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Copy shards Drive -> local disk (Drive FUSE is too slow for training I/O).
N_TRAIN_SHARDS = 200  # 1000 samples per shard; None = all
src = f'{DRIVE_ROOT}/data/full/shards'
dst = f'{LOCAL_DATA}/shards'
os.makedirs(dst, exist_ok=True)
names = sorted(os.listdir(src)) if os.path.isdir(src) else []
keep = [n for n in names if n.endswith('.tar') and (not n.startswith('train-') or
        N_TRAIN_SHARDS is None or int(n[6:12]) < N_TRAIN_SHARDS)]
t = time.time()
for n in keep:
    if not os.path.exists(f'{dst}/{n}'):
        subprocess.run(['cp', f'{src}/{n}', f'{dst}/{n}'], check=True)
print(f'{len(keep)} shards in {time.time() - t:.0f}s' if keep else 'no shards on Drive yet')
!du -sh {dst}

In [ ]:
# Real splits + nested label pools from the HF dataset (deterministic; see make_real_splits.py)
HF, REAL = '/content/hf', '/content/v1/data/real_v2'
os.makedirs(REAL, exist_ok=True)
if not os.path.isdir(f'{REAL}/pool2048'):
    !{PY} scripts/hf_to_manifest.py inspect --repo yaniguan/ocsr-dataset --local {HF}/ocsr
    !{PY} scripts/make_real_splits.py --local {HF}/ocsr --out {HF}/real_v2
    for m in ['eval', 'pool128', 'pool512', 'pool2048']:
        !{PY} scripts/ingest_real.py --manifest {HF}/real_v2/manifest_{m}.csv --out {REAL}/{m}
    !cp {HF}/real_v2/eval_keys.txt /content/v1/data/eval_keys_v2.txt
    !cp {HF}/real_v2/split_stats.json {DRIVE_ROOT}/v1/
!head -c 1500 {HF}/real_v2/split_stats.json

In [ ]:
# Layout expected by scripts/hpc/v1_task.py; runs and reports persist on Drive.
V1, V0, ORIG = '/content/v1', '/content/v0', '/content/orig'
for d in (V1, f'{V0}/data/full', ORIG, f'{DRIVE_ROOT}/v1/runs', f'{DRIVE_ROOT}/v1/reports'):
    os.makedirs(d, exist_ok=True)
def link(src, dst):
    if not os.path.lexists(dst):
        os.symlink(src, dst)
link(REPO_DIR, f'{V1}/code'); link('/content/venv', f'{ORIG}/venv')
link(f'{LOCAL_DATA}/shards', f'{V0}/data/full/shards')
link(f'{DRIVE_ROOT}/v1/runs', f'{V1}/runs'); link(f'{DRIVE_ROOT}/v1/reports', f'{V1}/reports')
!ls {V1} {V1}/data

In [ ]:
# Smoke gate: tests must pass on this exact checkout before spending GPU hours.
!{PY} -m pytest -q -x

## Run the grid

`TASKS` selects grid ids (print them with `!{PY} scripts/hpc/v1_grid.py`). Priority order when time is short: seed 0 first (ids 1-12), then seeds 1 and 2 (13-24, 25-36). A100 ~ 1-1.5 h per task. If the session is disconnected, reconnect, run all cells again; finished runs are skipped.

In [ ]:
import json, pathlib
TASKS = list(range(1, 13))  # seed 0, all arms and budgets
def finished(run_id):
    return pathlib.Path(f'{DRIVE_ROOT}/v1/runs/{run_id}/eval/summary.json').exists()
grid = [json.loads(subprocess.check_output([VENV_PY, 'scripts/hpc/v1_grid.py', '--task', str(i)]))
        for i in TASKS]
for t in grid:
    if finished(t['run_id']):
        print('skip', t['run_id']); continue
    print('RUN ', t['run_id'], time.strftime('%H:%M:%S'), flush=True)
    r = subprocess.run([VENV_PY, 'scripts/hpc/v1_task.py', '--task', str(t['id']),
                        '--original', ORIG, '--v0', V0, '--root', V1],
                       env=dict(os.environ, MPLBACKEND='Agg'))
    print('exit', r.returncode, 'finished' if finished(t['run_id']) else 'NOT finished', flush=True)
    if r.returncode:
        break  # look at the per-stage log in Drive/ouroboros/v1/reports before continuing

In [ ]:
for t in grid:
    p = pathlib.Path(f'{DRIVE_ROOT}/v1/runs/{t["run_id"]}/eval/summary.json')
    if p.exists():
        s = json.loads(p.read_text())
        print(t['run_id'], {k: s[k] for k in list(s)[:6]})